# Sampling-based planning with a toy world model

**Status: Illustrative state-space analogy.** This notebook compares simplified RRT and KPIECE tree search with categorical CEM shooting. Each planner uses the same hand-authored point-mass dynamics, paired starts, and maximum number of transition calls per planning decision.

The example uses CPU, fixed seeds, and no downloaded assets. It does not use images, DINO-WM, JEPA, a learned latent state, or a robot. Its transition function is the known toy environment itself.

We replan after a short action prefix. Each planning tree is discarded before the next decision. This makes the replanning structure easier to see; it is not an implementation of the cited paper.

## What the paper reports

Gao, Yang, and Kingston use frozen DINO-WM as a latent transition function. Their RRT selects nodes by nearest-neighbor distance. Their KPIECE variant favors low-coverage projected cells. Both use an action prior, batched expansions, and receding-horizon execution. They reset the tree after each replan and optionally polish a returned tree path with CEM.

The paper reports mixed success-rate comparisons and lower planning time in some settings. For example, its point_maze table reports RRT at 49/50 successes and 0.61 s median time, while CEM reports 50/50 and 16.07 s. In Push-T, RRT reports 40/50 and 0.72 s, while CEM reports 46/50 and 13.33 s. The table calculates time, cost, and transition-node summaries over solved trials. The paper also reports 29/30 versus 23/30 on its real-robot task. These are paper-reported values for its own model, task set, hardware, and protocol.

Sources: [paper v1](https://arxiv.org/abs/2610.12910), [full text and Table I](https://arxiv.org/html/2610.12910v1), and [author lab listing](https://commalab.org/publications/). The paper is listed as under review. The reviewed author pages link a PDF and video, but no official planner-code or checkpoint repository.

In [ ]:
import math
import platform
import sys
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

print(f'Python {platform.python_version()} | NumPy {np.__version__} | CPU')

# Fixed task and planner settings. These are tutorial settings, not paper settings.
GROUP_SEEDS = (2101, 2102, 2103, 2104, 2105)
STARTS_PER_GROUP = 20
HORIZON = 16
EXECUTE_PREFIX = 2
EPISODE_LIMIT = 80
TRANSITION_BUDGET = 256
CEM_POPULATION = 8
CEM_ITERATIONS = 2
GOAL_BIAS = 0.15
GOAL = np.array([1.0, 0.0])
OBSTACLE = (-0.22, 0.22, -0.46, 0.46)  # xmin, xmax, ymin, ymax
WORLD_LIMIT = 1.35
MAX_SPEED = 0.15
ACCELERATION = 0.035
CELL_WIDTH = 0.25

LEVELS = (-ACCELERATION, 0.0, ACCELERATION)
ACTIONS = np.array([(ax, ay) for ax in LEVELS for ay in LEVELS], dtype=float)
ZERO_ACTION_INDEX = int(np.flatnonzero(np.all(ACTIONS == 0.0, axis=1))[0])
PLANNERS = ('RRT', 'KPIECE', 'CEM')


## Define the task and transition rule

The state contains planar position and velocity. Nine discrete accelerations let the point mass move around a closed rectangular obstacle. The goal is beyond the obstacle. A transition is invalid when its swept segment touches the obstacle or leaves the world.

The planner uses this exact toy transition function. Therefore, this comparison has no learned-model error. This is a major difference from planning with a learned visual model.

In [ ]:
def segment_intersects_rectangle(p0, p1, rect):
    xmin, xmax, ymin, ymax = rect
    dx, dy = p1 - p0
    t_min, t_max = 0.0, 1.0
    for p, q in ((-dx, p0[0] - xmin), (dx, xmax - p0[0]),
                 (-dy, p0[1] - ymin), (dy, ymax - p0[1])):
        if p == 0.0:
            if q < 0.0:
                return False
            continue
        ratio = q / p
        if p < 0.0:
            t_min = max(t_min, ratio)
        else:
            t_max = min(t_max, ratio)
        if t_min > t_max:
            return False
    return True


def step(state, action):
    """One deterministic, hand-authored point-mass transition."""
    position = state[:2]
    velocity = np.clip(state[2:] + action, -MAX_SPEED, MAX_SPEED)
    next_position = position + velocity
    if np.any(np.abs(next_position) > WORLD_LIMIT):
        return None
    if segment_intersects_rectangle(position, next_position, OBSTACLE):
        return None
    return np.concatenate((next_position, velocity))


def reached_goal(state):
    return np.linalg.norm(state[:2] - GOAL) <= 0.16 and np.linalg.norm(state[2:]) <= 0.11


def sample_start(group_seed, start_id):
    rng = np.random.default_rng(group_seed * 100 + start_id)
    position = np.array([-1.0 + rng.uniform(-0.035, 0.035), rng.uniform(-0.06, 0.06)])
    velocity = rng.uniform(-0.01, 0.01, size=2)
    return np.concatenate((position, velocity))


def cell_for(state):
    return tuple(np.floor((state[:2] + WORLD_LIMIT) / CELL_WIDTH).astype(int))


def path_to_node(index, parents, actions):
    result = []
    while index > 0:
        result.append(actions[index])
        index = parents[index]
    return list(reversed(result))

# A simple local check of the declared task geometry.
assert step(np.array([-1.0, 0.0, 0.0, 0.0]), np.array([0.0, 0.0])) is not None
assert step(np.array([-0.24, 0.0, 0.02, 0.0]), np.array([0.02, 0.0])) is None
print('Task checks passed: free-space motion is valid and obstacle entry is rejected.')


## Implement the three planners

RRT and KPIECE share tree growth, the action set, goal test, and transition rule. RRT chooses the tree node nearest a sampled position. KPIECE chooses from a least-visited grid cell. These are compact teaching versions; they omit the paper's learned action prior, latent descriptors, and GPU batching.

CEM samples full action sequences and updates a categorical distribution from its best candidates. All planners have the same maximum of 256 one-step transition evaluations for each decision. The notebook also reports actual calls because a planner can stop early or reject a sequence.

In [ ]:
def tree_plan(start, rng, mode):
    # Fixed-size arrays avoid rebuilding the node matrix for every nearest query.
    states = np.empty((TRANSITION_BUDGET + 1, 4), dtype=float)
    positions = states[:, :2]
    parents = np.full(TRANSITION_BUDGET + 1, -1, dtype=int)
    action_ids = np.full(TRANSITION_BUDGET + 1, -1, dtype=int)
    depths = np.zeros(TRANSITION_BUDGET + 1, dtype=int)
    states[0] = start
    node_count = 1
    transitions = 0
    found = None
    cell_members = {cell_for(start): [0]}

    while transitions < TRANSITION_BUDGET:
        eligible = np.flatnonzero(depths[:node_count] < HORIZON)
        if len(eligible) == 0:
            break

        if mode == 'RRT':
            if rng.random() < GOAL_BIAS:
                target = GOAL.copy()
            else:
                target = rng.uniform(-WORLD_LIMIT, WORLD_LIMIT, size=2)
                for _ in range(10):
                    if not (OBSTACLE[0] <= target[0] <= OBSTACLE[1] and OBSTACLE[2] <= target[1] <= OBSTACLE[3]):
                        break
                    target = rng.uniform(-WORLD_LIMIT, WORLD_LIMIT, size=2)
            distances = np.linalg.norm(positions[eligible] - target, axis=1)
            parent = int(eligible[int(np.argmin(distances))])
        elif mode == 'KPIECE':
            active_cells = list(cell_members.items())
            least_visited = min(len(members) for _, members in active_cells)
            candidate_cells = [cell for cell, members in active_cells if len(members) == least_visited]
            chosen_cell = candidate_cells[int(rng.integers(len(candidate_cells)))]
            members = cell_members[chosen_cell]
            parent = members[int(rng.integers(len(members)))]
        else:
            raise ValueError(f'Unknown tree mode: {mode}')

        action_id = int(rng.integers(len(ACTIONS)))
        candidate = step(states[parent], ACTIONS[action_id])
        transitions += 1  # Invalid proposals also use one transition evaluation.
        if candidate is None:
            continue

        node_index = node_count
        states[node_index] = candidate
        parents[node_index] = parent
        action_ids[node_index] = action_id
        depths[node_index] = depths[parent] + 1
        node_count += 1
        if depths[node_index] < HORIZON:
            cell_members.setdefault(cell_for(candidate), []).append(node_index)
        if reached_goal(candidate):
            found = node_index
            break

    if found is not None:
        best = found
    elif node_count > 1:
        best = min(range(1, node_count), key=lambda i: np.linalg.norm(positions[i] - GOAL))
    else:
        return [ACTIONS[ZERO_ACTION_INDEX].copy()], transitions

    selected = []
    while best > 0:
        selected.append(int(action_ids[best]))
        best = int(parents[best])
    selected.reverse()
    return [ACTIONS[i].copy() for i in selected], transitions


def sequence_score(start, sequence):
    state = start.copy()
    calls = 0
    energy = 0.0
    for t, action_index in enumerate(sequence):
        action = ACTIONS[action_index]
        next_state = step(state, action)
        calls += 1
        energy += float(np.dot(action, action))
        if next_state is None:
            # Invalid trajectories receive a clear penalty and stop their rollout.
            return 4.0 + 0.01 * (HORIZON - t), calls
        state = next_state
    return float(np.linalg.norm(state[:2] - GOAL) + 0.02 * energy), calls


def cem_plan(start, rng):
    probabilities = np.full((HORIZON, len(ACTIONS)), 1.0 / len(ACTIONS))
    best_sequence = [ZERO_ACTION_INDEX] * HORIZON
    best_score = math.inf
    transitions = 0

    for _ in range(CEM_ITERATIONS):
        population = []
        scores = []
        for _ in range(CEM_POPULATION):
            sequence = [int(rng.choice(len(ACTIONS), p=probabilities[t])) for t in range(HORIZON)]
            score, calls = sequence_score(start, sequence)
            transitions += calls
            population.append(sequence)
            scores.append(score)
            if score < best_score:
                best_score, best_sequence = score, sequence

        elite_count = max(2, CEM_POPULATION // 4)
        elite_indices = np.argsort(scores)[:elite_count]
        for t in range(HORIZON):
            counts = np.bincount([population[i][t] for i in elite_indices], minlength=len(ACTIONS))
            empirical = counts / counts.sum()
            probabilities[t] = 0.25 * probabilities[t] + 0.75 * empirical

    assert transitions <= TRANSITION_BUDGET
    return [ACTIONS[i].copy() for i in best_sequence], transitions

assert CEM_POPULATION * CEM_ITERATIONS * HORIZON == TRANSITION_BUDGET


## Run the paired closed-loop comparison

There are five evaluation groups with 20 shared starts each. Each planner gets the same start in every group. We report all planned episodes, including normal failures and any captured execution errors. A group standard deviation describes variation across these five evaluation groups only; it is not training-seed variation.

In [ ]:
PLANNER_IDS = {'RRT': 11, 'KPIECE': 23, 'CEM': 37}

def run_episode(method, start, planner_seed):
    rng = np.random.default_rng(planner_seed)
    state = start.copy()
    executed_steps = 0
    model_calls = 0
    status = 'episode_limit'
    start_time = time.perf_counter()

    while executed_steps < EPISODE_LIMIT and not reached_goal(state):
        if method in ('RRT', 'KPIECE'):
            plan, calls = tree_plan(state, rng, method)
        else:
            plan, calls = cem_plan(state, rng)
        model_calls += calls
        if calls > TRANSITION_BUDGET:
            raise RuntimeError(f'{method} exceeded its per-decision transition budget')

        valid_action_executed = False
        for action in plan[:EXECUTE_PREFIX]:
            if executed_steps >= EPISODE_LIMIT:
                break
            next_state = step(state, action)
            if next_state is None:
                status = 'invalid_planned_action'
                break
            state = next_state
            executed_steps += 1
            valid_action_executed = True
            if reached_goal(state):
                status = 'success'
                break
        if status == 'invalid_planned_action':
            break
        if not valid_action_executed:
            status = 'no_executable_plan'
            break

    success = reached_goal(state)
    if success:
        status = 'success'
    return {
        'success': bool(success),
        'status': status,
        'steps': executed_steps,
        'transition_calls': model_calls,
        'runtime_seconds': time.perf_counter() - start_time,
        'final_x': float(state[0]),
        'final_y': float(state[1]),
    }

records = []
planned_per_method = len(GROUP_SEEDS) * STARTS_PER_GROUP
for group_index, group_seed in enumerate(GROUP_SEEDS):
    for start_id in range(STARTS_PER_GROUP):
        start = sample_start(group_seed, start_id)
        paired_id = f'g{group_index + 1}-s{start_id + 1:02d}'
        for method in PLANNERS:
            planner_seed = group_seed * 1000 + start_id * 41 + PLANNER_IDS[method]
            try:
                result = run_episode(method, start, planner_seed)
            except Exception as exc:
                result = {
                    'success': False, 'status': f'error:{type(exc).__name__}',
                    'steps': 0, 'transition_calls': 0, 'runtime_seconds': float('nan'),
                    'final_x': float(start[0]), 'final_y': float(start[1]),
                }
            records.append({
                'method': method, 'group': group_index + 1, 'group_seed': group_seed,
                'paired_start': paired_id, 'start_x': float(start[0]), 'start_y': float(start[1]),
                **result,
            })

print(f'Planned episodes per method: {planned_per_method}; records: {len(records)}')


In [ ]:
group_summary = []
for method in PLANNERS:
    for group in range(1, len(GROUP_SEEDS) + 1):
        subset = [row for row in records if row['method'] == method and row['group'] == group]
        group_summary.append({
            'method': method, 'group': group, 'episodes': len(subset),
            'success_rate': float(np.mean([row['success'] for row in subset])) if subset else float('nan'),
            'mean_steps': float(np.mean([row['steps'] for row in subset])) if subset else float('nan'),
            'mean_transition_calls': float(np.mean([row['transition_calls'] for row in subset])) if subset else float('nan'),
            'mean_runtime_seconds': float(np.nanmean([row['runtime_seconds'] for row in subset])) if subset else float('nan'),
            'errors': sum(row['status'].startswith('error:') for row in subset),
        })

method_summary = []
for method in PLANNERS:
    rows = [row for row in group_summary if row['method'] == method]
    episodes = sum(row['episodes'] for row in rows)
    group_rates = [row['success_rate'] for row in rows]
    completed = sum(row['method'] == method for row in records)
    statuses = dict(Counter(row['status'] for row in records if row['method'] == method))
    method_summary.append({
        'method': method, 'planned_episodes': episodes,
        'mean_group_success': float(np.mean(group_rates)),
        'sd_across_groups': float(np.std(group_rates, ddof=1)),
        'mean_steps': float(np.mean([row['mean_steps'] for row in rows])),
        'mean_transition_calls': float(np.mean([row['mean_transition_calls'] for row in rows])),
        'mean_runtime_seconds': float(np.mean([row['mean_runtime_seconds'] for row in rows])),
        'errors': sum(row['errors'] for row in rows), 'completed_records': completed,
        'missing_records': planned_per_method - completed, 'status_counts': statuses,
    })

print('Aggregate outcomes (success and standard deviation are across five groups):')
for row in method_summary:
    print({key: round(value, 3) if isinstance(value, float) else value
           for key, value in row.items()})
print('Per-group outcomes:')
for row in group_summary:
    print({key: round(value, 3) if isinstance(value, float) else value
           for key, value in row.items()})

assert all(row['planned_episodes'] == planned_per_method for row in method_summary)
assert all(row['completed_records'] == planned_per_method for row in method_summary)
assert all(row['missing_records'] == 0 for row in method_summary)
assert all(row['transition_calls'] >= 0 for row in records)
assert all(row['transition_calls'] <= TRANSITION_BUDGET * (EPISODE_LIMIT // EXECUTE_PREFIX + 1)
           for row in records)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Show the toy geometry and one common start.
ax = axes[0]
ax.add_patch(plt.Rectangle((OBSTACLE[0], OBSTACLE[2]), OBSTACLE[1] - OBSTACLE[0],
                            OBSTACLE[3] - OBSTACLE[2], color='0.25', label='closed obstacle'))
ax.scatter([-1.0], [0.0], marker='o', s=70, color='tab:blue', label='start')
ax.scatter([GOAL[0]], [GOAL[1]], marker='*', s=150, color='tab:green', label='goal')
ax.set(xlim=(-WORLD_LIMIT, WORLD_LIMIT), ylim=(-WORLD_LIMIT, WORLD_LIMIT),
       xlabel='x position', ylabel='y position', title='Shared detour task')
ax.set_aspect('equal')
ax.grid(alpha=0.2)
ax.legend(loc='lower right', fontsize=8)

# Compare paired group success rates without combining away group variation.
ax = axes[1]
for method in PLANNERS:
    subset = [row for row in group_summary if row['method'] == method]
    ax.plot([row['group'] for row in subset], [row['success_rate'] for row in subset],
            marker='o', label=method)
ax.set(xticks=range(1, len(GROUP_SEEDS) + 1), ylim=(-0.05, 1.05),
       xlabel='evaluation group', ylabel='success rate', title='Paired closed-loop outcomes')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

print(f'Python {platform.python_version()} | NumPy {np.__version__} | Matplotlib {__import__("matplotlib").__version__}')
print(f'CPU-only run | horizon={HORIZON} | prefix={EXECUTE_PREFIX} | per-decision cap={TRANSITION_BUDGET} one-step calls')


## Interpretation and limits

Use the numbers above only for this hand-authored task and these planner implementations. The tree planners can keep intermediate states that go around the obstacle. CEM searches complete sequences through a small categorical distribution. The result also depends on the selected budget, action set, grid width, horizon, goal bias, and random seeds.

This local comparison does not establish the paper's reported speed or success findings. It omits learned image features, DINO-WM predictions, the paper action prior, GPU batching, optional path polishing, paper task sets, and the physical robot. The exact paper evaluation remains **not run** because this review found no author-linked code or checkpoint repository. Revisit that gate if first-party artifacts and evaluation details become available.